# MMS Vibration Data Exploration

This notebook provides exploratory data analysis for MMS fault classification.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import load_mms_csv, build_dataset
from src.evaluation.visualization import plot_signal, plot_fft, plot_class_distribution

%matplotlib inline
sns.set_style('whitegrid')

print("Imports successful!")

## 1. Load Data

In [ ]:
# Load dataset
data_dir = Path('../data/raw')

file_map = {
    'normal': 'normal.csv',
    'unbalance_fault': 'unbalance_fault.csv',
    'misalignment_fault': 'misalignment_fault.csv',
    'bearing_fault': 'bearing_fault.csv'
}

X, y = build_dataset(data_dir, file_map)

print(f"Dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique classes: {np.unique(y)}")

## 2. Class Distribution

In [ ]:
# Plot class distribution
unique, counts = np.unique(y, return_counts=True)

plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(unique)), counts, color='steelblue', alpha=0.7)

for bar, count in zip(bars, counts):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., height,
             f'{count}\n({count/len(y)*100:.1f}%)',
             ha='center', va='bottom')

plt.xlabel('Fault Class')
plt.ylabel('Number of Samples')
plt.title('Class Distribution')
plt.xticks(range(len(unique)), unique, rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 3. Visualize Sample Signals

In [ ]:
# Plot one sample from each class
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

for i, fault_class in enumerate(np.unique(y)):
    # Get first sample of this class
    idx = np.where(y == fault_class)[0][0]
    signal = X[idx]
    
    # Plot X-axis only (for simplicity)
    axes[i].plot(signal[:, 0], linewidth=0.8, label='X-axis')
    axes[i].set_title(f'{fault_class} - Sample Signal (X-axis)', fontweight='bold')
    axes[i].set_ylabel('Amplitude')
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Sample')
plt.tight_layout()
plt.show()

## 4. FFT Analysis

In [ ]:
# Plot FFT for each class
sampling_rate = 5120  # Hz

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, fault_class in enumerate(np.unique(y)):
    # Get first sample of this class
    idx = np.where(y == fault_class)[0][0]
    signal = X[idx, :, 0]  # X-axis only
    
    # Compute FFT
    fft_vals = np.fft.rfft(signal)
    fft_mag = np.abs(fft_vals)
    fft_freq = np.fft.rfftfreq(len(signal), 1/sampling_rate)
    
    # Plot
    axes[i].plot(fft_freq, fft_mag, linewidth=0.8)
    axes[i].set_xlabel('Frequency (Hz)')
    axes[i].set_ylabel('Magnitude')
    axes[i].set_title(f'{fault_class} - FFT Spectrum')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim(0, 2000)  # Focus on 0-2000 Hz

plt.tight_layout()
plt.show()

## 5. Statistical Analysis

In [ ]:
# Compute statistics per class
stats = {}

for fault_class in np.unique(y):
    class_data = X[y == fault_class]
    
    stats[fault_class] = {
        'mean': class_data.mean(),
        'std': class_data.std(),
        'min': class_data.min(),
        'max': class_data.max(),
        'rms': np.sqrt(np.mean(class_data**2))
    }

# Display as DataFrame
stats_df = pd.DataFrame(stats).T
print("\nStatistical Summary by Class:")
print("=" * 60)
print(stats_df)

# Visualize
stats_df[['mean', 'std', 'rms']].plot(kind='bar', figsize=(10, 6))
plt.title('Statistical Metrics by Fault Class')
plt.ylabel('Value')
plt.xlabel('Fault Class')
plt.legend(title='Metric')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Data Quality Checks

In [ ]:
# Check for NaN and Inf values
print("Data Quality Checks:")
print("=" * 60)
print(f"Total samples: {len(X)}")
print(f"Shape: {X.shape}")
print(f"NaN values: {np.isnan(X).sum()}")
print(f"Inf values: {np.isinf(X).sum()}")
print(f"Data type: {X.dtype}")
print(f"Memory usage: {X.nbytes / (1024**2):.2f} MB")

# Check value ranges
print(f"\nValue Ranges:")
print(f"  Min: {X.min():.4f}")
print(f"  Max: {X.max():.4f}")
print(f"  Mean: {X.mean():.4f}")
print(f"  Std: {X.std():.4f}")

## 7. Correlation Analysis

In [ ]:
# Analyze correlation between channels
sample_signal = X[0]  # Take first sample

# Compute correlation matrix
correlation = np.corrcoef(sample_signal.T)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation,
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    xticklabels=['X', 'Y', 'Z'],
    yticklabels=['X', 'Y', 'Z'],
    vmin=-1,
    vmax=1,
    center=0
)
plt.title('Channel Correlation Matrix (Sample Signal)')
plt.tight_layout()
plt.show()

## Conclusion

This notebook provides basic exploration of the MMS vibration data. Key observations:

1. **Data is balanced** across all fault classes (if true)
2. **Signal characteristics** differ between fault types
3. **FFT spectra** show distinct frequency patterns for different faults
4. **Data quality** is good (no NaN or Inf values)
5. **Channels (X, Y, Z)** may show different levels of correlation

Next steps:
- Train models using `scripts/train.py`
- Evaluate performance
- Deploy via API